## Check GPU Availability

In [ ]:
!nvidia-smi

## Imports

In [ ]:
import torch
import wandb
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import load_dataset

from retentionengine.utils import DistillationTrainer

## Wandb setting

In [ ]:
PROJECT_NAME = "restoration_training"
RUN_NAME = "Qwen3_8B_adapter_restoration"

# WandB Initialization
wandb.init(project=PROJECT_NAME, name=RUN_NAME)

## Define Dataset

In [ ]:
# Output of: python scripts/dataset_generator.py --model-name Qwen/Qwen3-8B --output data/qwen3_memory_aligned_dataset.jsonl
dataset_path = "data/qwen3_memory_aligned_dataset.jsonl"

In [ ]:
dataset = load_dataset("json", data_files=dataset_path)

In [ ]:
dataset['train'][0]

In [ ]:
dataset['train'][0].keys()

## Load Model

In [ ]:
torch_dtype = torch.bfloat16

# Teacher: the original (untrimmed) model in bf16.
# FP8 is opt-in: pass use_fp8_teacher=True to DistillationTrainer (needs `retentionengine[fp8]`).
base_model_id = "Qwen/Qwen3-8B"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch_dtype,
)

# Student: the trimmed model saved by usage.ipynb (standard HF format)
student_model_id = "pretrained/Qwen3-8B-trimmed"
student_model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    torch_dtype=torch_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_function(examples):
    formatted_text = [
        f"질문: {inp}\n\n답변: {out}{tokenizer.eos_token}"
        for inp, out in zip(examples["input"], examples["target_output"])
    ]
    tokenized = tokenizer(formatted_text, truncation=True, padding="max_length", max_length=512)
    # Ignore padding only (attention_mask == 0): the real EOS stays a target even when pad == eos.
    tokenized["labels"] = [
        [tok if keep else -100 for tok, keep in zip(ids, mask)]
        for ids, mask in zip(tokenized["input_ids"], tokenized["attention_mask"])
    ]
    return tokenized

original_columns = dataset['train'].column_names
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=original_columns)

In [ ]:
split_dataset = tokenized_datasets["train"].train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

## Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen3-restored-final",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",                    # 매 에포크 종료 시 모델 체크포인트 저장
    logging_steps=10,                         # 10 스텝마다 로그 출력
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
)

# The student is a plain HF model, so the default remove_unused_columns=True is fine.
# If you pass a RetentionEngine instead, set remove_unused_columns=False in TrainingArguments.
trainer = DistillationTrainer(
    model=student_model,
    teacher_model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    alpha=1.0,
    temperature=0.3,
    use_fp8_teacher=False,
)

trainer.train()

# Save the restored student (best checkpoint) for the evaluation below
trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

wandb.finish()

## Testing

In [ ]:
# 파일명: testing.py

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import json

TEACHER_MODEL_ID = "Qwen/Qwen3-8B"
STUDENT_MODEL_PATH = "./qwen3-restored-final"
TEST_DATA_PATH = "data/qwen3_memory_aligned_dataset.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_TEST_SAMPLES = 50
MAX_NEW_TOKENS = 128

print("INFO: 모델 및 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
teacher_model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL_ID, torch_dtype=torch.bfloat16).to(DEVICE)
student_model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL_PATH, torch_dtype=torch.bfloat16).to(DEVICE)

teacher_model.eval()
student_model.eval()
print("INFO: 로딩 완료.")

test_prompts = []
try:
    with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= NUM_TEST_SAMPLES:
                break
            data = json.loads(line)
            prompt = f"질문: {data['input']}\n\n답변:"
            test_prompts.append(prompt)
except FileNotFoundError:
    print(f"ERROR: 테스트 데이터 파일을 찾을 수 없습니다. 경로를 확인하세요: {TEST_DATA_PATH}")
    raise

total_accuracy = 0
total_mismatched = 0
total_generated = 0

print(f"\nINFO: {len(test_prompts)}개의 샘플에 대한 테스트 시작...")
for sample_idx, prompt in enumerate(tqdm(test_prompts, desc="Testing Progress")):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        teacher_output_ids = teacher_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        student_output_ids = student_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    teacher_tokens = teacher_output_ids[0][inputs.input_ids.shape[1]:]
    student_tokens = student_output_ids[0][inputs.input_ids.shape[1]:]
    
    total_tokens_in_sample = len(teacher_tokens)
    if total_tokens_in_sample == 0:
        continue  # Teacher가 아무것도 생성하지 않은 경우 해당 샘플은 건너뜀

    mismatched_tokens_in_sample = 0
    mismatch_details = []

    len_to_compare = min(len(teacher_tokens), len(student_tokens))
    for i in range(len_to_compare):
        if teacher_tokens[i] != student_tokens[i]:
            mismatched_tokens_in_sample += 1
            mismatch_details.append(
                f"  - Position #{i+1}: "
                f"Teacher='{tokenizer.decode(teacher_tokens[i])}' (ID: {teacher_tokens[i]}) vs "
                f"Student='{tokenizer.decode(student_tokens[i])}' (ID: {student_tokens[i]})"
            )
            
    len_diff = abs(len(teacher_tokens) - len(student_tokens))
    if len_diff > 0:
        mismatched_tokens_in_sample += len_diff
        stop_reason = "Teacher가 먼저 중단" if len(teacher_tokens) < len(student_tokens) else "Student가 먼저 중단"
        mismatch_details.append(
            f"  - 생성 길이 불일치: Teacher={len(teacher_tokens)}개, Student={len(student_tokens)}개 ({stop_reason})"
        )

    if mismatch_details:
        print(f"\n\n--- [sample #{sample_idx+1}] doesn't match---")
        print(f"Prompt: \"{prompt[:80]}...\"")
        for detail in mismatch_details:
            print(detail)
        print(f"Teacher output: \"...{tokenizer.decode(teacher_tokens, skip_special_tokens=True)}\"")
        print(f"Student output: \"...{tokenizer.decode(student_tokens, skip_special_tokens=True)}\"")
        print("-" * 25)

    accuracy = ((total_tokens_in_sample - mismatched_tokens_in_sample) / total_tokens_in_sample) * 100
    total_accuracy += accuracy
    total_mismatched += mismatched_tokens_in_sample
    total_generated += total_tokens_in_sample

if len(test_prompts) > 0:
    avg_accuracy = total_accuracy / len(test_prompts)
    print("\n\n" + "="*30)
    print("--- 최종 평가 결과 요약 ---")
    print(f"평균 토큰 일치율 (정확도): {avg_accuracy:.2f}%")
    print(f"총 생성 토큰 (Teacher 기준): {total_generated}")
    print(f"총 불일치 토큰: {total_mismatched}")
    print("="*30)
else:
    print("테스트할 데이터를 찾지 못해 평가를 진행할 수 없습니다.")